PETS for Daily Budget Allocation
--------------------------------
 • Environment
     - 3 ad channels; fixed daily budget B.
     
     - Action  a_t  ∈ Δ²  (simplex of length‑3 => spend fractions).
     
     - Hidden “true” response function (unknown to the agent) with
       saturation & randomness.
       
     - State  s_t  = previous‑day conversions (3) + cumulative spend (3).
     
 • Agent
     - Probabilistic ensemble (N networks)  f_θ(s,a) → (Δμ, log σ²)
       predicts  next‑state  and  reward.
       
     - MPC with Cross‑Entropy Method (CEM) to optimise an action
       *sequence*  a_{t:t+H−1}  each step, but executes only  a_t.


In [1]:
# ────────────────────────────────────────────────────────────────────
# Imports & global config
# --------------------------------------------------------------------
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from collections import deque
from typing import Tuple, List

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
np.random.seed(0)
torch.manual_seed(0)

the thing that you looking for is market should be change based on conversions.

you can see we try to mimic real world budget env, in paid side view, budget will be set based on conv, spend and that is why we added these two metrics there.

main env componenet, init, reset, step

budget is assign beggining of each month...

env finish at the end of month

In [23]:
# ────────────────────────────────────────────────────────────────────
# 1.  Toy marketing environment (deterministic actions, stochastic state)
# --------------------------------------------------------------------

class BudgetEnv:
    """
    Very small, deterministic‑action, stochastic‑response environment:
      * Fixed daily budget B.
      * True channel efficiency decays with cumulative spend (saturation).
        """

    def __init__(self,
                 horizon_days = 30,
                 B = 1_000.0,
                 noise_std = 0.05):

        self.horizon = horizon_days
        self.B = B
        self.noise_std = noise_std

        # hidden “true” parameters (unknown to PETS)
        # hidden as we try to mimic real world env. and in real world these values are come from user and its unknown to model
         
        self.alpha = np.array([0.06, 0.05, 0.04])     # base CVR / efficiency
        self.beta  = np.array([0.85, 0.90, 0.87])     # diminishing‑returns shape
        self.gamma = np.array([5e-5, 5e-5, 5e-5])     # saturation w.r.t. cum. spend

        self.reset()

    # this function try to calculate conversion + noise, in real world it will be more simpler
    def _true_response(self, spend: np.ndarray):
        """Hidden conversion function with multiplicative noise."""
        # Saturation factor:  exp(‑γ * cumulative_spend)
        sat = np.exp(-self.gamma * self.cum_spend)
        mean_conv = self.alpha * (spend ** self.beta) * sat
        noise = np.random.normal(0.0, self.noise_std, size=spend.shape)
        return np.clip(mean_conv * (1.0 + noise), 0.0, None)

    def reset(self):
        self.day = 0
        self.cum_spend = np.zeros(3)
        self.prev_conv = np.zeros(3)
        return self._get_state()

    def _get_state(self):
        """
        State vector (6‑D):->[prev_conv_3, cum_spend_3]  
        (all normalised for NN stability)
        """

        # its your normalization, you are open to change it here
        norm_conv = self.prev_conv / (self.B * self.alpha)  # rough scale
        norm_cum = self.cum_spend / (self.horizon * self.B)
        return np.concatenate([norm_conv, norm_cum])



    def step(self, action: np.ndarray) -> Tuple[np.ndarray, float, bool, dict]:
        """
        action: 3‑dim vector(not necessarily summing to 1) -> action shows the amount we spend.
        We normalise to produce spend fractions.
        """

        frac = np.clip(action, 1e-6,None) 
        frac /= frac.sum() # make sure sum up to 1
        spend = frac * self.B # budget for each channel

        conv = self._true_response(spend)
        reward = conv.sum() # for future if one channel reward is more important, you can add weights

        self.day += 1
        self.cum_spend += spend
        self.prev_conv = conv.copy()

        done = self.day >= self.horizon
        return self._get_state(), float(reward), done, {}
        
        
        
        

In [25]:
class DynamicsModel(nn.Module):
    """
    Simple 2‑layer MLP with probabilistic output:
       Given (state, action) → predict Δstate and reward.
       Outputs mean  μ  and log variance  log σ²  for each target dimension.
    """
    def __init__(self, state_dim: int, action_dim: int, hidden: int = 128):
        super().__init__()
        self.fc1 = nn.Linear(state_dim + action_dim, hidden)
        self.fc2 = nn.Linear(hidden, hidden)
        self.fc_out = nn.Linear(hidden, state_dim + 1)     # (Δstate, reward)
        self.log_var = nn.Parameter(torch.zeros(state_dim + 1))  # global log σ²

    def forward(self, state, action):
        x = torch.cat([state, action], dim = -1)
        h = torch.relu(self.fc1(x))
        h = torch.relu(self.fc2(h))
        mu = self.fc_out(h)
        # broadcast global σ²
        log_var = self.log_var.expand_as(mu)
        return mu, log_var


 When is MSE OK?
 
You can use MSE (mean squared error) if and only if:

You assume the output noise is homoscedastic — i.e., constant variance for all data points.

Your model is not required to express uncertainty about its predictions.

$
y∼N(μ,σ^2),with fixed σ^2
$

----------------

In PETS, the goal is not just to predict accurately, but to:

Predict how uncertain the model is.

Use that uncertainty for:

Better planning (e.g. sampling trajectories with variability),

Exploration (avoid overly confident decisions in unknown states).

To train this kind of model, you need to maximize the likelihood, which gives you the Gaussian NLL loss:

$
[
\mathcal{L}_{\text{NLL}} = \frac{(y - \mu)^2}{\sigma^2} + \log \sigma^2
]
$

First term: Penalizes prediction error scaled by inverse variance.

Second term: Penalizes uncertainty (larger variance increases this term)



In [33]:
# ────────────────────────────────────────────────────────────────────
# 3.  Ensemble wrapper
# --------------------------------------------------------------------

class Ensemble:
    def __init__(self,
                 n_models,
                 state_dim,
                 action_dim,
                 lr = 1e-3,
                 weight_decay = 1e-4):
        
        self.n_models = n_models
        self.models = [DynamicsModel(state_dim, action_dim).to(DEVICE)
                       for _ in range(n_models)]
        
        self.optims = [optim.Adam(m.parameters(), lr = lr, weight_decay = weight_decay)
                       for m in self.models] #L2 reg, param = param - lr * (grad + weight_decay * param)      or        loss = loss + wd * paarm**2


    # gussian negative log likelyhood.
    #----------------------------------------------------------------
    def _gaussian_nll(mu, log_var, target):
        inv_var = torch.exp(-log_var)
        return ((mu - target) ** 2) * inv_var + log_var

    # ---------------------------------------------------------------
    def train(self,
              replay_buffer,
              batch_size = 256,
              epochs = 5):

        # bootstrap: each model trains on its own random resample
        data = np.array(replay_buffer, dtype = object)

        #train each model
        for m, optim_m in zip(self.models, self.optims):

            idx = np.random.choise(len(data), len(data), replace=True)

            for _ in epoch(idx):

                np.random.shuffle(idx)

                for start in range(0, len(idx), batch_size):

                    batch_idx = idx[start:start + batch_size]
                    s, a, r, s2 = zip(*data[batch_idx])
                    
                    s   = torch.tensor(np.stack(s),  dtype=torch.float32, device=DEVICE)
                    a   = torch.tensor(np.stack(a),  dtype=torch.float32, device=DEVICE)
                    r   = torch.tensor(np.array(r)[:,], dtype=torch.float32, device=DEVICE)
                    s2  = torch.tensor(np.stack(s2), dtype=torch.float32, device=DEVICE)  

                    # our target/ prediction is s2-s , r its more easier for model to learn and predict
                    target = torch.cat([s2 - s, r], dim=1)

                    mu, log_var = m(s, a)
                    loss = self._gaussian_nll(mu, log_var, target).mean()

                    optim_m.zero_grad()
                    loss.backward()
                    optim_m.step()


    def predict(self,
                state,
                action,
                model_idx):

        mu, log_var = self.models[model_idx](state,action)
        std = torch.exp(0.5 * log_var)
        eps = torch.randn_like(std)

        sampled = mu + eps * std                            # re‑param trick
        dstate = sampled[:, :-1]
        reward = sampled[:, -1:]
        next_state = state + dstate
        return next_state, reward
        

array([0, 3, 3], dtype=int32)